# Task 3: Cat vs. Dog Image Classifier

A small CNN trained on Kaggle's Dogs vs. Cats dataset. The goal is to build, inspect, and explain a baseline rather than chase the highest possible score.

## Dataset setup

Download `train.zip` from the Kaggle competition and extract its image files into `data/train/`. The files should be named like `cat.0.jpg` and `dog.0.jpg`.

In [ ]:
from pathlib import Path
import shutil
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

SEED = 42
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 3
BASE = Path.cwd()
RAW = BASE / 'data' / 'train'
SORTED = BASE / 'data' / 'sorted'
OUT = BASE / 'outputs'
OUT.mkdir(exist_ok=True)

if not RAW.exists():
    raise FileNotFoundError('Extract Kaggle train.zip into data/train first.')

for label in ['cat', 'dog']:
    target = SORTED / label
    target.mkdir(parents=True, exist_ok=True)
    if not any(target.iterdir()):
        for image in RAW.glob(f'{label}.*.jpg'):
            shutil.copy2(image, target / image.name)
print('Images ready:', {p.name: len(list(p.glob('*.jpg'))) for p in SORTED.iterdir()})

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    SORTED, validation_split=0.2, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='binary'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    SORTED, validation_split=0.2, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='binary'
)
class_names = train_ds.class_names
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
print('Class order:', class_names)

In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
])
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMG_SIZE + (3,)),
    augmentation,
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(32, 3, activation='relu'), tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'), tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'), tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
print(f'Final training accuracy: {train_acc:.3f}')
print(f'Final validation accuracy: {val_acc:.3f}')

plt.figure(figsize=(7, 4))
plt.plot(history.history['accuracy'], label='training accuracy')
plt.plot(history.history['val_accuracy'], label='validation accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Training curve'); plt.legend(); plt.grid(alpha=.25)
plt.tight_layout(); plt.savefig(OUT / 'learning_curve.png', dpi=180); plt.show()
(OUT / 'metrics.txt').write_text(f'Final training accuracy: {train_acc:.3f}\nFinal validation accuracy: {val_acc:.3f}\n')

In [ ]:

images, labels = next(iter(val_ds.unbatch().batch(32)))
probabilities = model.predict(images, verbose=0).ravel()
predictions = (probabilities >= .5).astype(int)
plt.figure(figsize=(15, 4))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    plt.imshow(images[i].numpy().astype('uint8'))
    confidence = probabilities[i] if predictions[i] == 1 else 1 - probabilities[i]
    plt.title(f'True: {class_names[int(labels[i])]}\nPred: {class_names[predictions[i]]} ({confidence:.0%})')
    plt.axis('off')
plt.tight_layout(); plt.show()

In [ ]:

wrong = np.where(predictions != labels.numpy().astype(int))[0]
if len(wrong):
    i = wrong[0]
    plt.figure(figsize=(4, 4)); plt.imshow(images[i].numpy().astype('uint8')); plt.axis('off')
    plt.title(f'Misclassified: true {class_names[int(labels[i])]}, predicted {class_names[predictions[i]]}\nConfidence: {max(probabilities[i], 1-probabilities[i]):.0%}')
    plt.show()
    print('Possible reason: the pose, background, low resolution, or partial visibility may make the animal resemble the other class.')
else:
    print('No mistakes in this first batch - run again with another validation batch to find one.')

## Brief conclusion

The printed final training and validation accuracy are the reported results for this run. A gap where training accuracy is higher than validation accuracy is a sign of overfitting; augmentation, dropout, and stopping after only a few epochs help keep that gap modest.